In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import plotly.offline as pyo
import plotly.graph_objs as go
import plotly.express as px
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from scipy.stats.mstats import winsorize
from sklearn.feature_selection import VarianceThreshold

In [3]:
df = pd.read_csv("Dataset/Harga Bahan Pangan/train/Daging Sapi Murni.csv")

In [4]:
def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [5]:
df.head()

,Date,Aceh,Bali,Banten,Bengkulu,DI Yogyakarta,DKI Jakarta,Gorontalo,Jambi,Jawa Barat,...,Papua,Riau,Sulawesi Barat,Sulawesi Selatan,Sulawesi Tengah,Sulawesi Tenggara,Sulawesi Utara,Sumatera Barat,Sumatera Selatan,Sumatera Utara
0,2022-01-01,150930.0,112770.0,124900.0,124090.0,124880.0,126580.0,125160.0,125410.0,122930.0,...,139560.0,134880.0,122940.0,119100.0,126720.0,125700.0,128680.0,130880.0,128970.0,133350.0
1,2022-01-02,151540.0,112020.0,126580.0,124910.0,124880.0,130000.0,123980.0,124910.0,123090.0,...,137050.0,136480.0,123940.0,119100.0,127360.0,124640.0,128840.0,129050.0,127630.0,132150.0
2,2022-01-03,150100.0,111770.0,124900.0,124290.0,124490.0,124900.0,125160.0,126670.0,122890.0,...,137110.0,136020.0,123280.0,118750.0,124840.0,124720.0,127830.0,128340.0,126640.0,131920.0
3,2022-01-04,148700.0,111090.0,126160.0,124910.0,124490.0,127940.0,125160.0,124900.0,122780.0,...,139710.0,136510.0,123280.0,118500.0,127600.0,124720.0,126690.0,129610.0,126960.0,133040.0
4,2022-01-05,147690.0,111240.0,127060.0,124290.0,124600.0,125900.0,124960.0,126200.0,122600.0,...,141280.0,135960.0,123280.0,119070.0,127600.0,125550.0,127250.0,128160.0,124390.0,129900.0


In [6]:
missing_percent_df = df.isnull().sum() / len(df) * 100
print(missing_percent_df)

Date                         0.000000
Aceh                         3.685259
Bali                         3.585657
Banten                       3.685259
Bengkulu                     3.784861
DI Yogyakarta                3.585657
DKI Jakarta                  3.685259
Gorontalo                    3.486056
Jambi                        3.784861
Jawa Barat                   3.685259
Jawa Tengah                  3.386454
Jawa Timur                   3.486056
Kalimantan Barat             3.585657
Kalimantan Selatan           3.784861
Kalimantan Tengah            3.585657
Kalimantan Timur             3.884462
Kalimantan Utara             3.884462
Kepulauan Bangka Belitung    3.784861
Kepulauan Riau               3.884462
Lampung                      3.685259
Maluku Utara                 3.585657
Maluku                       3.685259
Nusa Tenggara Barat          3.685259
Nusa Tenggara Timur          3.386454
Papua Barat                  3.884462
Papua                        3.685259
Riau        

In [7]:
numeric_features = df.select_dtypes(include=['number']).columns
df[numeric_features] = df[numeric_features].fillna(df[numeric_features].median())

In [8]:
zero_var_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Columns with zero variance:", zero_var_cols)

Columns with zero variance: []


In [9]:
missing_percent_df = df.isnull().sum() / len(df) * 100
print(missing_percent_df)

Date                         0.0
Aceh                         0.0
Bali                         0.0
Banten                       0.0
Bengkulu                     0.0
DI Yogyakarta                0.0
DKI Jakarta                  0.0
Gorontalo                    0.0
Jambi                        0.0
Jawa Barat                   0.0
Jawa Tengah                  0.0
Jawa Timur                   0.0
Kalimantan Barat             0.0
Kalimantan Selatan           0.0
Kalimantan Tengah            0.0
Kalimantan Timur             0.0
Kalimantan Utara             0.0
Kepulauan Bangka Belitung    0.0
Kepulauan Riau               0.0
Lampung                      0.0
Maluku Utara                 0.0
Maluku                       0.0
Nusa Tenggara Barat          0.0
Nusa Tenggara Timur          0.0
Papua Barat                  0.0
Papua                        0.0
Riau                         0.0
Sulawesi Barat               0.0
Sulawesi Selatan             0.0
Sulawesi Tengah              0.0
Sulawesi T

In [10]:
numerical_features = df.select_dtypes(include=['number'])
categorical_features = df.select_dtypes(exclude=['number'])

# Apply VarianceThreshold to remove low-variance numerical features
selector = VarianceThreshold(threshold=0.01)  # Adjust threshold as needed
reduced_numerical_df = selector.fit_transform(numerical_features)

# Convert back to DataFrame with selected features
reduced_numerical_df = pd.DataFrame(reduced_numerical_df, 
                                       columns=numerical_features.columns[selector.get_support()])

# Combine numerical and categorical features back together
reduced_df = pd.concat([reduced_numerical_df, categorical_features.reset_index(drop=True)], axis=1)

In [11]:
df.shape

(1004, 35)

In [12]:
reduced_df.shape

(1004, 35)

In [13]:
skewness = df.select_dtypes(include=['number']).apply(lambda x: stats.skew(x.dropna())).sort_values(ascending=False)
print(skewness)

Nusa Tenggara Barat          2.795809
Aceh                         2.791515
Sumatera Utara               2.064799
Sumatera Selatan             1.792979
Sulawesi Barat               1.669441
Banten                       0.625759
Jawa Barat                   0.486539
Jawa Timur                   0.444522
Lampung                      0.312766
Bali                        -0.112106
Papua Barat                 -0.161839
Bengkulu                    -0.277926
DKI Jakarta                 -0.279671
Papua                       -0.372260
Maluku Utara                -0.379493
Jambi                       -0.518685
Gorontalo                   -0.619546
Riau                        -0.664927
Kepulauan Riau              -0.671569
Nusa Tenggara Timur         -0.691804
Sulawesi Selatan            -0.719790
Kepulauan Bangka Belitung   -0.778950
Maluku                      -0.948054
Sulawesi Tenggara           -0.959370
Kalimantan Barat            -1.108553
Kalimantan Utara            -1.125786
Kalimantan S

In [14]:
# Select numerical columns
num_cols = df.select_dtypes(include=['number'])

# Function to calculate outlier percentage using IQR
def outlier_percentage(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((column < lower_bound) | (column > upper_bound)).sum()
    return (outliers / len(column)) * 100  # Percentage

# Apply function to all numerical columns
outlier_percentages_df = num_cols.apply(outlier_percentage)

# Display the results
print(outlier_percentages_df.sort_values(ascending=False))

Jawa Tengah                  16.633466
Jawa Barat                   16.633466
Aceh                         15.637450
Sumatera Selatan             14.940239
DI Yogyakarta                14.840637
Jambi                        14.840637
Sulawesi Selatan             14.840637
Kalimantan Tengah            14.342629
Kalimantan Barat             13.745020
Kalimantan Utara             13.346614
Sulawesi Utara               12.450199
Riau                         12.450199
Kalimantan Timur             11.752988
Lampung                      11.653386
Sumatera Barat               11.454183
Sulawesi Tengah              10.856574
Sulawesi Tenggara             9.362550
DKI Jakarta                   8.565737
Kepulauan Riau                8.366534
Banten                        8.366534
Bengkulu                      7.768924
Sumatera Utara                7.171315
Gorontalo                     6.175299
Nusa Tenggara Barat           5.278884
Nusa Tenggara Timur           4.183267
Kalimantan Selatan       

In [15]:
test = pd.read_csv("Dataset/Harga Bahan Pangan/test/Daging Sapi Murni.csv")

In [16]:
df.to_csv("Daging Sapi Murni Clean.csv", index=False)

In [17]:
def df_to_X_y(df, window_size=5):
  df_as_np = df.to_numpy()
  X = []
  y = []
  for i in range(len(df_as_np)-window_size):
    row = [[a] for a in df_as_np[i:i+window_size]]
    X.append(row)
    label = df_as_np[i+window_size]
    y.append(label)
  return np.array(X), np.array(y)

In [18]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, InputLayer
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping


In [19]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, InputLayer, Dropout, Conv1D, MaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split

# Load dataset
df = df.select_dtypes(include=[np.number])  # Keep only numeric columns
df = df.apply(pd.to_numeric, errors='coerce').dropna()  # Convert and drop NaNs

# If multiple numeric columns exist, use the first one
if df.shape[1] > 1:
    print(f"Warning: DataFrame has multiple numeric columns ({df.shape[1]}). Using the first column.")
    df = df.iloc[:, 0]

# Convert data into sequences
def df_to_X_y(df, window_size=10):  # Keep window size = 10 for CNN effectiveness
    X, y = [], []
    for i in range(len(df) - window_size):
        X.append(df[i:i+window_size].values)  # Directly use raw values
        y.append(df.iloc[i + window_size])  # Keep original values
    return np.array(X), np.array(y)

X, y = df_to_X_y(df, window_size=10)

# Reshape for CNN-LSTM (Conv1D requires 3D input)
X = X.reshape(X.shape[0], X.shape[1], 1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# Callbacks
checkpoint_path = "model_checkpoint.keras"
cp4 = ModelCheckpoint(filepath=checkpoint_path, save_best_only=True, monitor='val_loss', mode='min')
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.8, patience=5, min_lr=1e-6, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# Define CNN-LSTM model
def create_cnn_lstm_model(input_shape):
    model = Sequential([
        InputLayer(input_shape=input_shape),

        # CNN Layers
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),

        # LSTM Layers
        LSTM(128, return_sequences=True),
        Dropout(0.3),
        LSTM(64, return_sequences=True),
        Dropout(0.3),
        LSTM(32, return_sequences=False),
        Dropout(0.3),

        # Dense Layers
        Dense(16, activation='relu'),
        Dense(1, activation='linear')  # Output remains in original scale
    ])
    
    model.compile(loss='mape', optimizer=Adam(learning_rate=0.003), metrics=['mape'])
    return model

# Train the model
input_shape = (X_train.shape[1], X_train.shape[2])
model = create_cnn_lstm_model(input_shape)

history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
          epochs=300, batch_size=32, verbose=1, 
          callbacks=[cp4, lr_reducer, early_stopping])

print("Training completed. Final epoch:", len(history.history['loss']))

# Save the model
model.save("cnn_lstm_model.keras")

# Forecasting
df_submission = pd.read_csv("Dataset/Harga Bahan Pangan/sample_submission.csv")
unique_countries = df_submission['id'].str.split('/').str[1].unique()
total_required_predictions = 92 * len(unique_countries)

future_predictions = []
input_seq = X_test[-1]

for _ in range(total_required_predictions):
    pred = model.predict(input_seq.reshape(1, input_seq.shape[0], 1))[0, 0]
    pred += np.random.normal(0, 0.01)  # Add small noise to prevent stagnation
    future_predictions.append(pred)
    input_seq = np.roll(input_seq, -1)
    input_seq[-1] = pred

# Check if predictions match expected count
if len(future_predictions) != total_required_predictions:
    print(f"Warning: Expected {total_required_predictions} predictions but got {len(future_predictions)}")

# Save submission file
data = [{'id': df_submission.iloc[i]['id'], 'price': future_predictions[i]} for i in range(min(len(df_submission), len(future_predictions)))]
submission_df = pd.DataFrame(data)
submission_df.to_csv("Daging_Sapi_Murni_cnn_lstm_submission.csv", index=False)
print("Submission file saved as Daging_Sapi_Murni_cnn_lstm_submission.csv")

d:\Anaconda\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning:

Argument `input_shape` is deprecated. Use `shape` instead.



Epoch 1/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - loss: 99.9993 - mape: 99.9993 - val_loss: 99.9958 - val_mape: 99.9958 - learning_rate: 0.0030
Epoch 2/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 99.9946 - mape: 99.9946 - val_loss: 99.9907 - val_mape: 99.9907 - learning_rate: 0.0030
Epoch 3/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 99.9892 - mape: 99.9892 - val_loss: 99.9839 - val_mape: 99.9839 - learning_rate: 0.0030
Epoch 4/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 99.9817 - mape: 99.9817 - val_loss: 99.9750 - val_mape: 99.9750 - learning_rate: 0.0030
Epoch 5/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 99.9718 - mape: 99.9718 - val_loss: 99.9638 - val_mape: 99.9638 - learning_rate: 0.0030
Epoch 6/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 99.9602 - mape: 99.9602 - val_loss: 99.9504 - val_mape: 99.9504 - learning_rate: 0.0030
Epoch 7/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 99.9462 - mape: 99.9462 - val_loss: 99.9347 - val_ma